# Preprocessing of the "Raw" Data

In [1]:
import pandas as pd
import random
import string
import re
random.seed(42)

In [2]:
raw_df = pd.read_csv("../AP1/issue_df.csv")
print(raw_df.head())
# print(len(raw_df[raw_df["Party"] == "R"])/len(raw_df))
prop_republican = len(raw_df[raw_df["Party"] == "R"])/len(raw_df)
prop_dem = 1 - prop_republican

               Name    State District Party  \
0  Aderholt, Robert  Alabama      4th     R   
1  Aderholt, Robert  Alabama      4th     R   
2  Aderholt, Robert  Alabama      4th     R   
3  Aderholt, Robert  Alabama      4th     R   
4  Aderholt, Robert  Alabama      4th     R   

                                             Content  \
0  I believe the majority of Alabamians support s...   
1  "Knowledge is power." Thomas Jefferson, Januar...   
2  Energy becomes a more important topic each and...   
3  "Congress shall make no law respecting an esta...   
4  The Founding Fathers wisely included the 2nd A...   

                                          Issue Link  
0          https://aderholt.house.gov/issues/economy  
1        https://aderholt.house.gov/issues/education  
2           https://aderholt.house.gov/issues/energy  
3  https://aderholt.house.gov/issues/faith-and-re...  
4       https://aderholt.house.gov/issues/gun-rights  


In [3]:
name_to_anonymized = {}
clean_df = raw_df.copy()
for name in set(raw_df["Name"]):
    clean_df.loc[clean_df["Name"] == name, "Name"] = "".join(
        random.choices(string.ascii_uppercase + string.digits, k=10)
    )
clean_df.drop(columns=["State", "District", "Issue Link"], inplace=True)
clean_df.head()

,Name,Party,Content
0,EYUHZ1GV0E,R,I believe the majority of Alabamians support s...
1,EYUHZ1GV0E,R,"""Knowledge is power."" Thomas Jefferson, Januar..."
2,EYUHZ1GV0E,R,Energy becomes a more important topic each and...
3,EYUHZ1GV0E,R,"""Congress shall make no law respecting an esta..."
4,EYUHZ1GV0E,R,The Founding Fathers wisely included the 2nd A...


In [4]:
old_length = len(clean_df)
for name, party, content in zip(clean_df["Name"], clean_df["Party"], clean_df["Content"]):
    sentence_content = content.split(".")
    if len(sentence_content) >= 3:
        for i in range(len(sentence_content) - 2):
            three_sentence = sentence_content[i] + "." + sentence_content[i + 1] + "." + sentence_content[i + 2]
            clean_df.loc[len(clean_df)] = [name, party, three_sentence]
        else:
            clean_df.loc[len(clean_df)] = [name, party, None]
clean_df = clean_df.dropna()
clean_df = clean_df.iloc[old_length:]
clean_df.reset_index(inplace=True)

clean_df

,index,Name,Party,Content
0,1330,EYUHZ1GV0E,R,I believe the majority of Alabamians support s...
1,1331,EYUHZ1GV0E,R,"We need to protect taxpayers, ensure our seni..."
2,1332,EYUHZ1GV0E,R,We have seen continually weak growth in the ec...
3,1333,EYUHZ1GV0E,R,"The time for action is now. In May 2012, I vo..."
4,1334,EYUHZ1GV0E,R,"In May 2012, I voted to replace the sequester..."
...,...,...,...,...
21368,24021,HEJLA0GNZS,R,has delegated authority to unelected bureaucr...
21369,24022,HEJLA0GNZS,R,\nAdministrative agencies have the power to w...
21370,24023,HEJLA0GNZS,R,How this translates in Wyoming is the EPA’s a...
21371,24025,HEJLA0GNZS,R,"As an attorney, I fought to return control of ..."


In [12]:
sample_size = 100
republican_samples = clean_df[clean_df["Party"] == "R"].sample(n=round(sample_size*prop_republican), random_state=42)
democratic_samples = clean_df[clean_df["Party"] == "D"].sample(n=sample_size - round(sample_size*prop_republican), random_state=42)
ap3_df = pd.concat([republican_samples, democratic_samples]).reset_index(drop=True)
ap3_df = ap3_df.sample(frac=1, random_state=42).reset_index(drop=True)
ap3_df.drop(columns=["Party"], inplace=True)
ap3_df["Label"] = None
ap3_df.rename(columns={"index": "Unique Data Point ID"}, inplace=True)
ap3_df.head()

,Unique Data Point ID,Name,Content,Label
0,11211,3FPOYIPK0Q,_____________________________________________...,None
1,15890,KBP2QD6VAS,These brave men and women and their families ...,None
2,16236,25SQ8CROYR,"In New Jersey, there are 23 federally-funded ...",None
3,4854,DPG8SBI4Q2,The Inflation Reduction Actcreated the Medica...,None
4,8106,UK9E1V2ISQ,gov. The Department of Consumer Protection has...,None


In [13]:
ap3_df.to_csv("ap3_df.tsv", sep="\t", index=False)